# Introduction to Usage of LLM for custom usecases
### Practice Notebook

**Assumed pre-installed libraries:** `numpy`, `scikit-learn`

This notebook is a conceptual, hands-on companion to the Day 1 lecture. Since we
don't have a live LLM API key wired into this environment, we simulate the
"generate" step of RAG with a simple template function.

Wherever you see
`# TODO: replace with a real LLM call`, that's where you'd plug in the
Groq in a real project — the retrieval logic around it stays
the same.


In [8]:
import os
from groq import Groq

In [9]:
groq_key=os.environ["GROQ_API_KEY"]

## 1. Why retrieval matters

Three core motivations for RAG (Retrieval-Augmented Generation):

1. **Knowledge cutoff** — an LLM only knows what was in its training data. It
   cannot answer questions about events, documents, or data created after
   that cutoff, or about anything that was never public.
2. **Factual grounding** — LLMs generate the statistically likely next token,
   not a database lookup. Without an external source to check against, they
   can produce fluent, confident, and *wrong* answers (hallucination).
3. **Domain-specific / private data** — a general-purpose LLM was never
   trained on your company's internal wiki, your product manuals, or your
   customer records. RAG lets the model answer questions about that data
   without retraining it.

Run the cell below to see the failure mode RAG is designed to fix: asking a
model a question it has no way of knowing the answer to, with nothing to
ground it.


In [1]:
def naive_llm_answer(question: str) -> str:
    """Stand-in for a raw LLM call with NO retrieval step.
    A real model would still generate *something* here -- often a fluent,
    confident, wrong answer. We simulate that with a canned response.
    """
    return (
        "I believe the answer is approximately correct based on general "
        "patterns I learned during training, but I have no specific source "
        "for this and may be fabricating details. (This is what an "
        "ungrounded hallucination looks like.)"
    )

question = "What was our company's Q3 refund policy exception for order #48213?"
print("Question:", question)
print("Ungrounded answer:\n", naive_llm_answer(question))


Question: What was our company's Q3 refund policy exception for order #48213?
Ungrounded answer:
 I believe the answer is approximately correct based on general patterns I learned during training, but I have no specific source for this and may be fabricating details. (This is what an ungrounded hallucination looks like.)


**Discussion:** the model above has no way to know about order #48213 — that
data doesn't exist in any public training set. This is exactly the gap RAG
closes: retrieve the relevant document first, then let the model generate
*from* that document instead of from memory alone.


## 2. RAG architecture: Retrieve → Augment → Generate

```
   User question
        |
        v
 [1. RETRIEVE]  ---->  search a knowledge base (vector DB, keyword index, etc.)
        |                for the most relevant chunks of text
        v
 [2. AUGMENT]   ---->  insert those chunks into the prompt as context
        |
        v
 [3. GENERATE]  ---->  the LLM answers using the question + the retrieved context
        |
        v
   Grounded answer (ideally with citations back to the source chunks)
```

Let's build a minimal toy version of this pipeline end to end, using a tiny
in-memory "knowledge base" and a simple keyword-overlap retriever (we'll
upgrade this to real embeddings on Day 3).


## How RAG is different than fine tuning?

In [ ]:
# A tiny knowledge base -- in a real system this would be thousands of
# document chunks stored in a vector database.
knowledge_base = [
    {"id": "doc1", "text": "Our Q3 refund policy allows exceptions for orders "
                            "delayed more than 10 business days, approved by a "
                            "supervisor. Order #48213 was delayed 14 days and "
                            "was granted a full refund on 2025-08-02."},
    {"id": "doc2", "text": "Standard refund policy: refunds are issued within "
                            "5-7 business days of a return being received."},
    {"id": "doc3", "text": "Our shipping partners include BlueDart, Delhivery, "
                            "and DTDC for domestic orders across India."},
]


In [4]:
def simple_keyword_retrieve(question: str, kb: list, top_k: int = 1) -> list:
    """Very naive retrieval: score each doc by how many question words it
    contains. This is NOT how production RAG retrieval works (see Day 3's
    embedding-based semantic search) -- it's here purely so you can see the
    RETRIEVE step in isolation, without any ML library required.
    """
    q_words = set(question.lower().split())
    scored = []
    for doc in kb:
        doc_words = set(doc["text"].lower().split())
        overlap = len(q_words & doc_words)
        scored.append((overlap, doc))
    print(scored)
    scored.sort(key=lambda x: x[0], reverse=True)
    return [doc for score, doc in scored[:top_k] if score > 0]

retrieved = simple_keyword_retrieve(question, knowledge_base, top_k=1)
for d in retrieved:
    print(f"Retrieved [{d['id']}]:", d["text"])


[(7, {'id': 'doc1', 'text': 'Our Q3 refund policy allows exceptions for orders delayed more than 10 business days, approved by a supervisor. Order #48213 was delayed 14 days and was granted a full refund on 2025-08-02.'}), (1, {'id': 'doc2', 'text': 'Standard refund policy: refunds are issued within 5-7 business days of a return being received.'}), (2, {'id': 'doc3', 'text': 'Our shipping partners include BlueDart, Delhivery, and DTDC for domestic orders across India.'})]
Retrieved [doc1]: Our Q3 refund policy allows exceptions for orders delayed more than 10 business days, approved by a supervisor. Order #48213 was delayed 14 days and was granted a full refund on 2025-08-02.


## What if the retrieved keywords are very few? Does that mean that the sentences are similar?

In [ ]:
def augment_prompt(question: str, retrieved_docs: list) -> str:
    """The AUGMENT step: build a prompt that gives the model the retrieved
    context and instructs it to answer only from that context.
    """
    context = "\n\n".join(f"[{d['id']}] {d['text']}" for d in retrieved_docs)
    prompt = (
        "Answer the question using ONLY the context below. If the context "
        "doesn't contain the answer, say you don't know.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\n"
        "Answer:"
    )
    return prompt

prompt = augment_prompt(question, retrieved)
print(prompt)


Answer the question using ONLY the context below. If the context doesn't contain the answer, say you don't know.

Context:
[doc1] Our Q3 refund policy allows exceptions for orders delayed more than 10 business days, approved by a supervisor. Order #48213 was delayed 14 days and was granted a full refund on 2025-08-02.

Question: What was our company's Q3 refund policy exception for order #48213?
Answer:


In [11]:
def generate_answer(prompt: str) -> str:
    """Stand-in GENERATE step. Replace this with a real call, e.g.:

    # TODO: replace with a real LLM call"""
    client = Groq(api_key=groq_key)
    response = client.chat.completions.create(
        model="groq/compound",
        max_tokens=300,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content
    """
    For this offline practice notebook we just extract the most relevant
    sentence from the context as a stand-in for generation.
    """
    context_line = prompt.split("Context:\n")[1].split("\n\nQuestion:")[0]
    return f"(stubbed generation) Based on the retrieved context: {context_line.strip()}"

print(generate_answer(prompt))


**Answer:** The exception for order #48213 under the Q3 refund policy was that it received a **full refund**.

**Reasoning based on the context**

- The Q3 refund policy states that an exception can be made for any order delayed **more than 10 business days**, provided a supervisor approves it.  
- Order #48213 was delayed **14 days**, which exceeds the 10‑day threshold.  
- The context explicitly says that this order “was granted a full refund on 2025‑08‑02,” indicating the supervisor approved the exception.  

Therefore, the policy exception applied to order #48213 was the granting of a full refund.


**Exercise 1.1:** Add a new document to `knowledge_base` about a topic of your
choice, then ask a question that only that document can answer. Confirm the
retrieval step picks it up.

**Exercise 1.2:** Try a question with *no* relevant document in the knowledge
base (e.g., "What's the weather today?"). What does `simple_keyword_retrieve`
return? What should the final system do when retrieval comes back empty?
(Hint: this connects directly to Day 5's grounding/hallucination material.)


## 3. RAG vs. fine-tuning vs. long-context

| Approach | Best for | Weakness |
|---|---|---|
| **RAG** | Frequently changing / large knowledge bases; need citations; cheap to update (just re-index) | Retrieval quality bottlenecks the whole system; extra infra (vector DB) |
| **Fine-tuning** | Teaching a model a *style*, *format*, or *skill* (not facts); stable, narrow domains | Expensive to retrain for every data update; can't easily cite sources; risk of forgetting |
| **Long-context** | Small, fixed document sets that fit in the context window; simplicity | Cost/latency scale with context size; "needle in a haystack" retrieval can degrade with very long contexts; doesn't scale to huge corpora |

**Exercise 1.3:** For each of these scenarios, decide which approach (RAG,
fine-tuning, or long-context) fits best and justify why in one sentence:
1. A legal team wants a chatbot that answers questions from a 50,000-document
   contract archive, updated weekly.
2. A support team wants the model to always respond in a specific brand voice
   and ticket format.
3. A student wants to ask questions about a single 40-page PDF textbook
   chapter.
